# Thyroid Disease Detection Model

## Loading the dataset

In [ ]:
from utils import load_thyroid_data_3_classes
X, y = load_thyroid_data_3_classes()

/media/DIURNOext4/alejandro/wip-clase/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /home/avidaldo/.cache/kagglehub/datasets/emmanuelfwerr/thyroid-disease-data/versions/2


## Definición de la métrica de evaluación

## Modelo con Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline

pipeline_random_forest = make_pipeline(
    preprocessor,
    RandomForestClassifier(random_state=42)
)

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold

stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores_random_forest = cross_val_score(pipeline_random_forest, X_train, y_train, 
                                      cv=stratified_cv, scoring=thyroid_scorer)

print(f'Random Forest Disease Recall: {scores_random_forest.mean():.3f}')

### Optimización de hiperparámetros

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_dist = {
    'randomforestclassifier__n_estimators': randint(100, 350),
    'randomforestclassifier__max_depth': randint(10, 110),
    'randomforestclassifier__min_samples_split': randint(2, 10),
    'randomforestclassifier__min_samples_leaf': randint(1, 4),
    'randomforestclassifier__max_features': ['sqrt', 'log2', None]
}

rnd_search_rforest = RandomizedSearchCV(
    estimator=pipeline_random_forest, 
    param_distributions=param_dist, 
    scoring=thyroid_scorer,
    cv=stratified_cv, 
    n_iter=20, 
    random_state=42, 
    verbose=1,
    n_jobs=-1
)

_ = rnd_search_rforest.fit(X_train, y_train)

In [ ]:
print(f"Best score for Random Forest: {rnd_search_rforest.best_score_}")

## Otro modelo: XGBoost

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
le.fit(y_train)  # Fit the encoder on the training data
y_train_encoded = le.transform(y_train)
y_test_encoded = le.transform(y_test)

In [ ]:
def custom_f2_score(y_true, y_pred):
    # Get original class names from the LabelEncoder
    all_classes = le.classes_
    # Calculate recall for each class using original labels
    recalls = recall_score(le.inverse_transform(y_true), 
                          le.inverse_transform(y_pred), 
                          labels=all_classes, 
                          average=None, 
                          zero_division=0)
    # Get indices for disease classes
    disease_indices = [np.where(all_classes == cls)[0][0] 
                      for cls in ['hyperthyroid', 'hypothyroid']]
    return np.mean(recalls[disease_indices])

thyroid_scorer2 = make_scorer(custom_f2_score)  # TODO: unificar con el de arriba

from xgboost import XGBClassifier
pipeline_xgb = make_pipeline(preprocessor, XGBClassifier(random_state=42))

scores_xgb = cross_val_score(pipeline_xgb, X_train, y_train_encoded, 
                            cv=stratified_cv, scoring=thyroid_scorer2)

print(f'XGBoost Disease Recall: {scores_xgb.mean():.3f}')

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

# Define the parameter grid to search
param_grid = {
    'xgbclassifier__n_estimators': np.arange(50, 500, 50),
    'xgbclassifier__max_depth': np.arange(3, 15, 1),
    'xgbclassifier__learning_rate': np.logspace(-3, 0, 10),
    'xgbclassifier__subsample': np.arange(0.5, 1.0, 0.1),
    'xgbclassifier__colsample_bytree': np.arange(0.5, 1.0, 0.1),
    'xgbclassifier__gamma': np.arange(0, 1.0, 0.1),
    'xgbclassifier__min_child_weight': np.arange(1, 10, 1)
}


# Set up the randomized search
rnd_search_xgb = RandomizedSearchCV(
    estimator=pipeline_xgb,
    param_distributions=param_grid,
    n_iter=100,  # Number of parameter settings sampled
    scoring=thyroid_scorer2,  # Or 'f1', 'roc_auc', etc.
    cv=stratified_cv,  
    verbose=1,
    random_state=42,
    n_jobs=-1  # Use all available cores
)

_ = rnd_search_xgb.fit(X_train, y_train_encoded)

In [ ]:
print(f"Best score for XGBoost: {rnd_search_xgb.best_score_}")